# 04-4. 최종 데이터 품질 재검토

04-3 최종 데이터를 수정하지 않고 남은 품질 후보를 유형별 CSV로 저장한다. 길이와 숫자 규칙은 자동 삭제 기준이 아니라 수동 검토 후보이다.

In [1]:
import re, sys
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT=Path(r"C:\Users\KDS-30\Desktop\Projects\06_Team_Project4")
assert Path(sys.prefix).resolve()==(PROJECT/".venv").resolve(), "커널을 Python 3.11 (Team4 News GPU)로 바꾸세요."
INPUT=PROJECT/"processed_data"/"04-3_final"
RESULTS=PROJECT/"review_data"/"04-4_final_quality_review"
assert INPUT.exists(), "먼저 04-3을 실행하세요."
RESULTS.mkdir(parents=True,exist_ok=True)
word_pattern=re.compile(r"[가-힣A-Za-z0-9]+(?:[.,]\d+)?")


In [2]:
df=pd.concat([pd.read_csv(INPUT/f"{s}.csv") for s in ["train","validation","test"]],ignore_index=True)
def words(text): return word_pattern.findall(str(text).lower())
def max_4gram(text):
    tokens=words(text); counts=Counter(tuple(tokens[i:i+4]) for i in range(len(tokens)-3))
    return max(counts.values(),default=0)
def overlap(document,summary):
    source=set(words(document)); target=words(summary)
    return np.nan if not target else sum(token in source for token in target)/len(target)
def numbers(text):
    text=re.sub(r"(?<=\d)\s+(?=\d)","",str(text)).replace(",","")
    return set(re.findall(r"\d+(?:\.\d+)?",text))
df["document_words"]=df.document.map(lambda x:len(words(x))); df["document_chars"]=df.document.str.len(); df["summary_chars"]=df.summary.str.len()
df["char_compression_ratio"]=df.summary_chars/df.document_chars
df["max_same_4gram_occurrences"]=df.summary.map(max_4gram); df["repeat_4gram_remaining"]=df.max_same_4gram_occurrences.ge(3)
df["summary_document_overlap"]=[overlap(d,s) for d,s in zip(df.document,df.summary)]; df["low_overlap"]=df.summary_document_overlap.lt(.30)
df["extreme_ratio"]=(df.char_compression_ratio.lt(.02)|df.char_compression_ratio.gt(.80)); df["summary_longer"]=df.summary_chars.gt(df.document_chars)
df["short_document"]=df.document_words.lt(30); df["short_and_summary_longer"]=df.short_document & df.summary_longer
doc_numbers=df.document.map(numbers); sum_numbers=df.summary.map(numbers); df["numbers_only_in_summary"]=[", ".join(sorted(s-d)) for d,s in zip(doc_numbers,sum_numbers)]; df["number_mismatch"]=df.numbers_only_in_summary.ne("")
df["remaining_url"]=df.document.str.contains(r"https?://|www\.",case=False,regex=True)
df["remaining_boilerplate"]=df.document.str.contains(r"무단\s*(?:전재|복제|배포)|재배포\s*금지|인터뷰\s*자료의\s*저작권|실제\s*방송\s*내용과\s*차이",regex=True)
df["remaining_replacement_char"]=df.document.str.contains("�",regex=False)
assert not df.repeat_4gram_remaining.any(); assert not df.remaining_url.any(); assert not df.remaining_replacement_char.any()


In [3]:
BASE=["row_id","split","assigned_split","source_index","date","category","press","title","link","document","summary","document_words","document_chars","summary_chars","char_compression_ratio","summary_document_overlap","numbers_only_in_summary"]
RULES={"01_repeat_4gram_remaining":df.repeat_4gram_remaining,"02_low_overlap":df.low_overlap,"03_extreme_ratio":df.extreme_ratio,"04_summary_longer":df.summary_longer,"05_short_document":df.short_document,"06_short_and_summary_longer":df.short_and_summary_longer,"07_number_mismatch":df.number_mismatch,"08_remaining_boilerplate":df.remaining_boilerplate}
def save(frame,path):
    try: frame.to_csv(path,index=False,encoding="utf-8-sig"); return path.name
    except PermissionError:
        fallback=path.with_name(path.stem+"_latest"+path.suffix); frame.to_csv(fallback,index=False,encoding="utf-8-sig"); return fallback.name
rows=[]
for name,mask in RULES.items():
    out=df.loc[mask,BASE].copy(); out["review_decision"]=""; out["drop_reason"]=""; out["memo"]=""
    filename=save(out,RESULTS/f"{name}.csv"); rows.append({"file":filename,"rows":len(out)})
summary=pd.DataFrame(rows); save(summary,RESULTS/"00_export_summary.csv")
flag_cols=["extreme_ratio","summary_longer","short_document","short_and_summary_longer","number_mismatch","remaining_boilerplate"]
flagged=df[df[flag_cols].any(axis=1)][BASE+flag_cols].copy(); flagged["matched_rules"]=flagged[flag_cols].apply(lambda r:", ".join(r.index[r].tolist()),axis=1); flagged["review_decision"]=""; flagged["memo"]=""
save(flagged,RESULTS/"00_all_remaining_flagged_rows.csv")
pd.DataFrame([{"input_data":str(INPUT),"input_rows":len(df),"policy":"review only; no rows removed in 04-4"}]).to_csv(RESULTS/"00_run_info.csv",index=False,encoding="utf-8-sig")
display(summary); print("통합 후보:",len(flagged))

,file,rows
0,01_repeat_4gram_remaining.csv,0
1,02_low_overlap.csv,0
2,03_extreme_ratio.csv,1024
3,04_summary_longer.csv,610
4,05_short_document.csv,486
5,06_short_and_summary_longer.csv,414
6,07_number_mismatch.csv,643
7,08_remaining_boilerplate.csv,2


통합 후보: 1635
